<a href="https://colab.research.google.com/github/NNwobi-354/semiarid-land-degradation-trajectory/blob/main/Nigeria_Kalmykia_Land_Degradation_ML_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CLONE SEMI-ARID LAND DEGRADATION RESEARCH REPOSITORY
# ============================================================

import os

# GitHub repository
GITHUB_REPO = "https://github.com/NNwobi-354/semiarid-land-degradation-trajectory.git"

# Clone repository into Colab
!git clone {GITHUB_REPO}

# Repository path
REPO_PATH = "/content/semiarid-land-degradation-trajectory"

# Check that the repository exists
if os.path.exists(REPO_PATH):
    print("✅ Repository successfully cloned!")
    print(f"📁 Repository location: {REPO_PATH}")
else:
    print("❌ Repository was not cloned successfully.")

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Exploratory Data Analysis & Quality Audit for Desertification Panels
# PROJECT: Semi-Arid Land Degradation Trajectory (2003–2025)
# DATASETS: GLDAS-2.2 / MODIS / CHIRPS Longitudinal Panels
# PATHS:
#   1. /content/semiarid-land-degradation-trajectory/Data/CSV_Files/BornoYobe_2003_2025.csv
#   2. /content/semiarid-land-degradation-trajectory/Data/CSV_Files/Kalmykia_2003_2025.csv
# ==============================================================================
# PURPOSE:
# This script performs an exhaustive data diagnostic across both regional CSVs
# to guarantee error-free downstream econometric modeling (Panel Fixed Effects,
# Spatial Regressions, and Machine Learning Trajectory Predictions).
#
# AUDIT STEPS PERFORMED:
# 1. Structural Verification: Row counts, column types, memory footprint.
# 2. Longitudinal Balance Check: Year coverage and grid-cell balance over time.
# 3. Missingness Audit: Detection of NaN/Null values per variable.
# 4. Descriptive Statistics: Mean, SD, Median, IQR, Min, Max for key predictors.
# 5. Outlier & Anomaly Detection: Z-score & physical range boundary checks.
# 6. Multi-Region Comparison Summary: Side-by-side distribution inspects.
# ==============================================================================

import os
import pandas as pd

# Target dataset paths
file_paths = [
    "/content/semiarid-land-degradation-trajectory/Data/CSV_Files/BornoYobe_2003_2025.csv",
    "/content/semiarid-land-degradation-trajectory/Data/CSV_Files/Kalmykia_2003_2025.csv"
]

def inspect_dataset(file_path):
    filename = os.path.basename(file_path)
    print("=" * 80)
    print(f" DATASET REPORT: {filename}")
    print("=" * 80)

    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}\n")
        return

    # Load dataset
    df = pd.read_csv(file_path)

    # 1. Dimensions & Duplicates
    rows, cols = df.shape
    duplicates = df.duplicated().sum()
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    print(f"\n📌 Overview:")
    print(f"  • Total Rows: {rows:,}")
    print(f"  • Total Columns: {cols:,}")
    print(f"  • Duplicate Rows: {duplicates:,}")
    print(f"  • Memory Usage: {memory_mb:.2f} MB")

    # 2. Column-level diagnostics
    print("\n📌 Column Diagnostics:")
    diag_df = pd.DataFrame({
        "Data Type": df.dtypes,
        "Non-Null": df.notnull().sum(),
        "Missing": df.isnull().sum(),
        "Missing (%)": (df.isnull().sum() / rows * 100).round(2),
        "Unique Values": df.nunique()
    })
    print(diag_df.to_string())

    # 3. Head & Tail Sample
    print("\n📌 Data Preview (First 3 Rows):")
    print(df.head(3).to_string())

    # 4. Statistical Summary for Numeric Data
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        print("\n📌 Statistical Summary (Numeric Columns):")
        stats = df[numeric_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
        stats.columns = ['Count', 'Mean', 'Std Dev', 'Min', 'Median (50%)', 'Max']
        print(stats.round(4).to_string())

    # 5. Categorical / Date / String Preview
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(categorical_cols) > 0:
        print("\n📌 Categorical Column Sample Values:")
        for col in categorical_cols:
            top_samples = df[col].dropna().value_counts().head(3).to_dict()
            print(f"  • {col}: {top_samples}")

    print("\n" + "=" * 80 + "\n")

# Run analysis on all target files
for path in file_paths:
    inspect_dataset(path)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Data Cleaning, Quality Remediation & Feature Engineering Pipeline
# PROJECT: Semi-Arid Land Degradation Trajectory Analysis (2003–2025)
# DATASETS: Integrated MODIS & ERA5-Land Longitudinal Datasets
# PATHS:
#   1. /content/semiarid-land-degradation-trajectory/Data/CSV_Files/BornoYobe_2003_2025.csv
#   2. /content/semiarid-land-degradation-trajectory/Data/CSV_Files/Kalmykia_2003_2025.csv
# ==============================================================================
# PURPOSE:
# This script cleans physical anomalies identified during the data quality audit,
# filters non-vegetated pixels, engineers comparative hydro-climatic indicators,
# and outputs standardized panel datasets ready for Machine Learning (XGBoost/SHAP)
# and spatial-temporal trend analyses (Mann-Kendall / Sen's Slope).
#
# PROCESSING STEPS PERFORMED:
# 1. Non-Vegetated Pixel Filtering: Removes non-vegetated or body-of-water pixels
#    (Target_NDVI <= 0.05) to prevent baseline distortion in dryland dynamics.
# 2. Flux Anomaly Rectification: Clips negative reanalysis artifacts in AET,
#    Latent Heat, and Soil Evaporation (values < 0 clamped to 0).
# 3. Hydro-Climatic Feature Engineering: Computes the Atmospheric-to-Subsurface
#    Demand-Supply Ratio (VPD / SMroot) to proxy vegetation moisture stress.
# 4. Cross-Biome Standardization: Computes within-biome Z-scores for predictors
#    to enable direct comparative model attribution across distinct climate regimes.
# 5. Clean Dataset Export: Saves cleaned files as 'BornoYobe_Cleaned.csv' and
#    'Kalmykia_Cleaned.csv' in the working environment.
# ==============================================================================

import pandas as pd
import numpy as np
import os

# Define input file paths
INPUT_PATHS = {
    "BornoYobe": "/content/semiarid-land-degradation-trajectory/Data/CSV_Files/BornoYobe_2003_2025.csv",
    "Kalmykia": "/content/semiarid-land-degradation-trajectory/Data/CSV_Files/Kalmykia_2003_2025.csv"
}

OUTPUT_DIR = "/content"
cleaned_datasets = {}

print("=" * 80)
print(" STARTING DATA CLEANING & PREPROCESSING PIPELINE")
print("=" * 80)

for region_name, file_path in INPUT_PATHS.items():
    print(f"\n Processing: {region_name}")
    print(f" Source: {file_path}")

    # Read raw dataset
    df = pd.read_csv(file_path)
    initial_rows = len(df)

    # --------------------------------------------------------------------------
    # STEP 1: Filter Non-Vegetated Pixels & Water Bodies
    # --------------------------------------------------------------------------
    df = df[df['Target_NDVI'] > 0.05].copy()
    filtered_rows = len(df)
    removed_pixels = initial_rows - filtered_rows
    print(f"  • Filtered {removed_pixels:,} non-vegetated rows (Target_NDVI <= 0.05).")

    # --------------------------------------------------------------------------
    # STEP 2: Rectify Reanalysis Physical Anomalies (Negative Fluxes)
    # --------------------------------------------------------------------------
    flux_cols = ['X5_AET_Annual', 'X6_Latent_Heat_LE', 'X7_Soil_Evaporation']
    for col in flux_cols:
        neg_count = (df[col] < 0).sum()
        df[col] = df[col].clip(lower=0.0)
        if neg_count > 0:
            print(f"  • Clamped {neg_count:,} negative values in '{col}' to 0.0.")

    # --------------------------------------------------------------------------
    # STEP 3: Hydro-Climatic Feature Engineering
    # --------------------------------------------------------------------------
    # Demand-to-Supply Moisture Stress Index: VPD / SMroot
    # Adding 1e-5 to prevent potential division by zero
    df['VPD_to_SM_Ratio'] = df['X3_Vapor_Deficit_VPD'] / (df['X8_Soil_Moisture_SMroot'] + 1e-5)
    print("  • Engineered feature: 'VPD_to_SM_Ratio' (Atmospheric Demand / Soil Supply).")

    # --------------------------------------------------------------------------
    # STEP 4: Standardized Features (Z-Scores for Cross-Biome Comparison)
    # --------------------------------------------------------------------------
    predictor_cols = [
        'X1_Rain_P_antecedent', 'X2_Max_Temp_Tmax', 'X3_Vapor_Deficit_VPD',
        'X5_AET_Annual', 'X6_Latent_Heat_LE', 'X7_Soil_Evaporation',
        'X8_Soil_Moisture_SMroot', 'X9_Rainfall_Anomaly', 'VPD_to_SM_Ratio'
    ]

    for col in predictor_cols:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df[f'{col}_zscore'] = (df[col] - mean_val) / std_val

    print(f"  • Computed within-region Z-scores for {len(predictor_cols)} predictor variables.")

    # --------------------------------------------------------------------------
    # STEP 5: Export Cleaned Datasets
    # --------------------------------------------------------------------------
    out_path = os.path.join(OUTPUT_DIR, f"{region_name}_Cleaned.csv")
    df.to_csv(out_path, index=False)
    cleaned_datasets[region_name] = df

    print(f"  ✓ Saved cleaned dataset: {out_path} [{len(df):,} rows x {df.shape[1]} cols]")

print("\n" + "=" * 80)
print(" PIPELINE COMPLETE: Datasets are ready for downstream ML & modeling.")
print("=" * 80)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Dataset Inspection & Structural Diagnostic Pipeline
# PURPOSE: Extract exact column names, shapes, missing values, and summary stats
#          to ensure downstream scripts match the cleaned dataset schema perfectly.
# ==============================================================================

import pandas as pd
import numpy as np
import os

# Define file paths for both raw and cleaned datasets
DATASETS = {
    "BornoYobe Cleaned": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia Cleaned": "/content/Kalmykia_Cleaned.csv"
}

print("=" * 85)
print(" DATASET STRUCTURAL & STATISTICAL DIAGNOSTIC REPORT")
print("=" * 85)

for dataset_name, file_path in DATASETS.items():
    print(f"\n{'-' * 85}")
    print(f" DATASET: {dataset_name}")
    print(f" PATH:    {file_path}")
    print(f"{'-' * 85}")

    if not os.path.exists(file_path):
        print(f" ERROR: File not found at {file_path}. Please check path.")
        continue

    # Load dataset
    df = pd.read_csv(file_path)

    # 1. basic Metadata
    print(f"\n[1] DIMENSIONS:")
    print(f"    • Total Rows:    {df.shape[0]:,}")
    print(f"    • Total Columns: {df.shape[1]}")

    # 2. Detailed Column Inventory (Types & Nulls)
    print(f"\n[2] COLUMN INVENTORY & NULL COUNTS:")
    col_info = pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Non-Null Count": df.notnull().sum(),
        "Null Count": df.isnull().sum(),
        "Null Percentage": (df.isnull().sum() / len(df) * 100).round(2).astype(str) + "%"
    })
    print(col_info.to_string())

    # 3. Sample Data Preview
    print(f"\n[3] FIRST 3 ROWS PREVIEW:")
    print(df.head(3).to_string())

    # 4. Statistical Summary for Numerical Columns
    print(f"\n[4] STATISTICAL SUMMARY (Min, Mean, Max, Std):")
    num_summary = df.describe().T[['min', 'mean', 'max', 'std']]
    print(num_summary.to_string())

print("\n" + "=" * 85)
print(" DIAGNOSTIC COMPLETE: Share this log to build error-free analytical scripts.")
print("=" * 85)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Section 3.1 Complete Pipeline (3.1.1, 3.1.2, 3.1.3)
# MANUSCRIPT: Spatial-Temporal Trajectories and Structural Breakpoints
# PUBLICATION LAYOUT: Heavy academic styling (680 DPI, 4.0pt spines, 3.5pt ticks)
# ENVIRONMENT: Python 3.10+ (Pandas, NumPy, SciPy, Matplotlib, Seaborn)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. ACADEMIC PUBLICATION STYLING CONFIGURATION
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

def apply_heavy_academic_style(ax):
    """Enforces heavy publication aesthetics across all figure axes."""
    for spine in ax.spines.values():
        spine.set_color('#000000')
        spine.set_linewidth(4.0)
        spine.set_visible(True)

    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=8, labelsize=12)
    ax.tick_params(axis='both', which='minor', colors='#000000', width=2.0, length=4)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', zorder=0)

def style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2):
    """Formats high-contrast bold legends positioned outside plot bounds to prevent overlap."""
    legend = ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#000000', fontsize=11,
                       loc=loc, bbox_to_anchor=bbox_to_anchor, ncol=ncol)
    legend.get_frame().set_linewidth(2.5)
    for text in legend.get_texts():
        text.set_fontweight('bold')
    return legend

# ------------------------------------------------------------------------------
# 2. HELPER STATISTICAL FUNCTIONS
# ------------------------------------------------------------------------------
def calc_sens_slope(x, y):
    """Non-parametric Sen's slope estimator."""
    n = len(x)
    slopes = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            if x[j] != x[i]:
                slopes.append((y[j] - y[i]) / (x[j] - x[i]))
    return np.median(slopes) if len(slopes) > 0 else 0.0

def pettitt_test(series):
    """Non-parametric Pettitt test for single abrupt structural breakpoint detection."""
    data = np.asarray(series)
    T = len(data)
    U_list = []
    for t in range(1, T):
        sub1 = data[:t]
        sub2 = data[t:]
        U = np.sum([np.sign(x1 - x2) for x1 in sub1 for x2 in sub2])
        U_list.append(U)

    K = np.max(np.abs(U_list))
    t_break_idx = np.argmax(np.abs(U_list)) + 1
    p_val = 2.0 * np.exp((-6.0 * (K**2)) / (T**3 + T**2))
    return t_break_idx, K, p_val

# ------------------------------------------------------------------------------
# 3. DATASET PATHS & SETUP
# ------------------------------------------------------------------------------
DATASETS = {
    "Borno & Yobe": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia": "/content/Kalmykia_Cleaned.csv"
}

# Column mappings
TARGET_COL = 'Target_NDVI'
VPD_COL = 'X3_Vapor_Deficit_VPD'
SM_COL = 'X8_Soil_Moisture_SMroot'
TIME_COL = 'Year'

spatial_dfs = {}
sec311_summary = []
sec312_summary = []
sec313_summary = []
regional_annual_means = {}

print("=" * 85)
print(" STARTING COMPLETE SECTION 3.1 ANALYTICAL PIPELINE")
print("=" * 85)

# ------------------------------------------------------------------------------
# 4. SECTION 3.1.1: PIXEL-LEVEL GREENING & BROWNING DYNAMICS
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.1.1: PIXEL-LEVEL MANN-KENDALL & SEN'S SLOPE ANALYSES")
print("#" * 85)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)
    pixel_records = []
    grouped = df.groupby(['Longitude', 'Latitude'])

    for (lon, lat), group in grouped:
        group_sorted = group.sort_values(TIME_COL)
        x_vals = group_sorted[TIME_COL].values
        y_vals = group_sorted[TARGET_COL].values

        if len(y_vals) >= 5:
            tau, p_val = stats.kendalltau(x_vals, y_vals)
            slope = calc_sens_slope(x_vals, y_vals)

            if p_val < 0.05 and slope > 0:
                trend_cat = 'Significant Greening'
            elif p_val < 0.05 and slope < 0:
                trend_cat = 'Significant Browning'
            else:
                trend_cat = 'Non-Significant Shift'

            pixel_records.append({
                'Longitude': lon, 'Latitude': lat, 'Tau': tau,
                'P_Value': p_val, 'Sens_Slope': slope, 'Trend_Category': trend_cat
            })

    res_df = pd.DataFrame(pixel_records)
    spatial_dfs[region_name] = res_df

    total_px = len(res_df)
    sig_green = (res_df['Trend_Category'] == 'Significant Greening').sum() / total_px * 100
    sig_brown = (res_df['Trend_Category'] == 'Significant Browning').sum() / total_px * 100
    non_sig = (res_df['Trend_Category'] == 'Non-Significant Shift').sum() / total_px * 100

    sec311_summary.append({
        'Region': region_name,
        'Mean Sen Slope (NDVI yr⁻¹)': res_df['Sens_Slope'].mean(),
        'Significant Greening (%)': sig_green,
        'Significant Browning (%)': sig_brown,
        'Non-Significant (%)': non_sig
    })

df_311 = pd.DataFrame(sec311_summary)
print("\n--- SECTION 3.1.1 SUMMARY TABLE: SURFACE AREA DYNAMICS ---")
print(df_311.to_string(index=False))

# --- FIGURE 1: Greening vs. Browning Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 7), dpi=680)
cats = ['Significant Greening\n(p < 0.05)', 'Significant Browning\n(p < 0.05)', 'Non-Significant Shift\n(p ≥ 0.05)']
x = np.arange(len(cats))
w = 0.35

r1 = df_311[df_311['Region'] == 'Borno & Yobe'].iloc[0]
r2 = df_311[df_311['Region'] == 'Kalmykia'].iloc[0]

v1 = [r1['Significant Greening (%)'], r1['Significant Browning (%)'], r1['Non-Significant (%)']]
v2 = [r2['Significant Greening (%)'], r2['Significant Browning (%)'], r2['Non-Significant (%)']]

rects1 = ax.bar(x - w/2, v1, w, label='Borno & Yobe', color='#2ca02c', edgecolor='#000000', linewidth=3.0, zorder=3)
rects2 = ax.bar(x + w/2, v2, w, label='Kalmykia', color='#d62728', edgecolor='#000000', linewidth=3.0, zorder=3)

# Inline Labeling with buffer offset above bars
for rect in rects1:
    h = rect.get_height()
    ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, h), xytext=(0, 6),
                textcoords="offset points", ha='center', va='bottom', fontsize=12, fontweight='bold', color='#000000')

for rect in rects2:
    h = rect.get_height()
    ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, h), xytext=(0, 6),
                textcoords="offset points", ha='center', va='bottom', fontsize=12, fontweight='bold', color='#000000')

ax.set_ylabel('Proportion of Total Surface Area (%)', fontsize=14, fontweight='bold', color='#000000')
ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=12, fontweight='bold')
ax.set_ylim(0, max(max(v1), max(v2)) * 1.25)

apply_heavy_academic_style(ax)
style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('Fig1_Greening_Browning_Area_Comparison.png', dpi=680, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 5. SECTION 3.1.2: LONG-TERM HYDRO-CLIMATIC TRENDS (VPD & SM_root)
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.1.2: LONG-TERM HYDRO-CLIMATIC EVOLUTION (VPD vs. SM_root)")
print("#" * 85)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)
    annual = df.groupby(TIME_COL)[[TARGET_COL, VPD_COL, SM_COL]].mean().reset_index()
    regional_annual_means[region_name] = annual

    vpd_slope = calc_sens_slope(annual[TIME_COL].values, annual[VPD_COL].values)
    sm_slope = calc_sens_slope(annual[TIME_COL].values, annual[SM_COL].values)
    _, vpd_p = stats.kendalltau(annual[TIME_COL].values, annual[VPD_COL].values)
    _, sm_p = stats.kendalltau(annual[TIME_COL].values, annual[SM_COL].values)

    sec312_summary.append({
        'Region': region_name,
        'VPD Trend (kPa yr⁻¹)': vpd_slope,
        'VPD p-value': vpd_p,
        'SM_root Trend (m³ m⁻³ yr⁻¹)': sm_slope,
        'SM_root p-value': sm_p
    })

df_312 = pd.DataFrame(sec312_summary)
print("\n--- SECTION 3.1.2 SUMMARY TABLE: HYDRO-CLIMATIC TRENDS ---")
print(df_312.to_string(index=False))

# --- FIGURE 2: Dual-Axis Atmospheric Demand (VPD) & Water Supply (SM_root) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), dpi=680, sharex=True)

for ax, reg in zip([ax1, ax2], ["Borno & Yobe", "Kalmykia"]):
    ann = regional_annual_means[reg]
    years = ann[TIME_COL].values

    # Left axis: VPD
    line1 = ax.plot(years, ann[VPD_COL], color='#d62728', linewidth=4.5, marker='o', markersize=8, label='VPD (Atmospheric Demand)', zorder=3)
    ax.set_ylabel('VPD (kPa)', fontsize=13, fontweight='bold', color='#d62728')
    ax.tick_params(axis='y', colors='#d62728')

    v_min, v_max = ann[VPD_COL].min(), ann[VPD_COL].max()
    ax.set_ylim(v_min - (v_max - v_min) * 0.1, v_max + (v_max - v_min) * 0.25)

    # Right axis: Soil Moisture
    ax_sm = ax.twinx()
    line2 = ax_sm.plot(years, ann[SM_COL], color='#1f77b4', linewidth=4.5, linestyle='--', marker='s', markersize=8, label='SM_root (Water Supply)', zorder=3)
    ax_sm.set_ylabel('SM_root (m³ m⁻³)', fontsize=13, fontweight='bold', color='#1f77b4')
    ax_sm.tick_params(axis='y', colors='#1f77b4')

    sm_min, sm_max = ann[SM_COL].min(), ann[SM_COL].max()
    ax_sm.set_ylim(sm_min - (sm_max - sm_min) * 0.1, sm_max + (sm_max - sm_min) * 0.25)

    # Non-overlapping Inline Annotations
    ax.annotate(f'Latest VPD: {ann[VPD_COL].iloc[-1]:.2f} kPa', xy=(years[-1], ann[VPD_COL].iloc[-1]),
                xytext=(-120, 15), textcoords='offset points', fontsize=10, fontweight='bold', color='#d62728',
                bbox=dict(boxstyle="round,pad=0.2", fc="#ffffff", ec="#d62728", lw=1.5))

    apply_heavy_academic_style(ax)
    apply_heavy_academic_style(ax_sm)

    ax.text(0.02, 0.88, reg, transform=ax.transAxes, fontsize=13, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="#ffffff", ec="#000000", lw=2.0))

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    leg = ax.legend(lines, labels, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=True, facecolor='#ffffff', edgecolor='#000000')
    leg.get_frame().set_linewidth(2.0)
    for t in leg.get_texts(): t.set_fontweight('bold')

ax2.set_xlabel('Year', fontsize=14, fontweight='bold', color='#000000')
plt.tight_layout()
plt.savefig('Fig2_HydroClimatic_Trends_VPD_SM.png', dpi=680, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 6. SECTION 3.1.3: ABRUPT SHIFT IDENTIFICATION (PETTITT BREAKPOINT TEST)
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.1.3: PETTITT REGIME SHIFT & BREAKPOINT ANALYSIS")
print("#" * 85)

for region_name in DATASETS.keys():
    ann = regional_annual_means[region_name]
    years = ann[TIME_COL].values
    ndvi_vals = ann[TARGET_COL].values

    break_idx, K_stat, p_val = pettitt_test(ndvi_vals)
    break_year = years[break_idx]

    pre_mean = np.mean(ndvi_vals[:break_idx])
    post_mean = np.mean(ndvi_vals[break_idx:])
    shift_pct = ((post_mean - pre_mean) / pre_mean) * 100

    sec313_summary.append({
        'Region': region_name,
        'Break Year': break_year,
        'Pettitt K': K_stat,
        'p-value': p_val,
        'Pre-Break Mean NDVI': pre_mean,
        'Post-Break Mean NDVI': post_mean,
        'Shift (%)': shift_pct
    })

df_313 = pd.DataFrame(sec313_summary)
print("\n--- SECTION 3.1.3 SUMMARY TABLE: PETTITT BREAKPOINTS & BASELINE SHIFTS ---")
print(df_313.to_string(index=False))

# --- FIGURE 3: Temporal Breakpoint Dynamics with Baseline Shift Overlay (ENLARGED CANVAS) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), dpi=680, sharey=True)

for ax, region_name in zip(axes, DATASETS.keys()):
    ann = regional_annual_means[region_name]
    years = ann[TIME_COL].values
    ndvi_vals = ann[TARGET_COL].values

    b_info = df_313[df_313['Region'] == region_name].iloc[0]
    b_year = int(b_info['Break Year'])

    ax.plot(years, ndvi_vals, color='#000000', linewidth=4.5, marker='o', markersize=8, label='Annual Mean NDVI', zorder=3)
    ax.axvline(b_year, color='#d62728', linestyle='--', linewidth=3.5, label=f'Breakpoint ({b_year})', zorder=4)

    ax.hlines(b_info['Pre-Break Mean NDVI'], years[0], b_year - 1, colors='#1f77b4', linestyles='-', linewidth=4.0,
              label=f'Pre-Break Mean ({b_info["Pre-Break Mean NDVI"]:.3f})', zorder=5)
    ax.hlines(b_info['Post-Break Mean NDVI'], b_year, years[-1], colors='#2ca02c', linestyles='-', linewidth=4.0,
              label=f'Post-Break Mean ({b_info["Post-Break Mean NDVI"]:.3f})', zorder=5)

    y_min, y_max = np.min(ndvi_vals), np.max(ndvi_vals)
    ax.set_ylim(y_min - (y_max - y_min) * 0.15, y_max + (y_max - y_min) * 0.25)

    # Non-overlapping Shift Info Box
    ax.annotate(f"{region_name}\nShift: {b_info['Shift (%)']:+.2f}%\n(p = {b_info['p-value']:.4f})",
                xy=(0.04, 0.72), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=12, fontweight='bold', color='#000000', zorder=6)

    ax.set_xlabel('Year', fontsize=14, fontweight='bold', color='#000000')
    apply_heavy_academic_style(ax)
    style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

axes[0].set_ylabel('NDVI', fontsize=14, fontweight='bold', color='#000000')
plt.tight_layout()
plt.savefig('Fig3_Pettitt_Regime_Shift_NDVI.png', dpi=680, bbox_inches='tight')
plt.show()

print("\n" + "=" * 85)
print(" SCRIPT COMPLETE: All Section 3.1 analyses finished and figures saved at 680 DPI.")
print("=" * 85)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Section 3.2 Complete Pipeline (3.2.1, 3.2.2)
# MANUSCRIPT: Spatial-Temporal Trajectories and Structural Breakpoints
# PUBLICATION LAYOUT: Heavy academic styling (680 DPI, 4.0pt spines, 3.5pt ticks)
# ENVIRONMENT: Python 3.10+ (Pandas, NumPy, SciPy, Matplotlib, Scikit-Learn, XGBoost)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. ACADEMIC PUBLICATION STYLING CONFIGURATION
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

def apply_heavy_academic_style(ax):
    """Enforces heavy publication aesthetics across all figure axes."""
    for spine in ax.spines.values():
        spine.set_color('#000000')
        spine.set_linewidth(4.0)
        spine.set_visible(True)

    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=8, labelsize=12)
    ax.tick_params(axis='both', which='minor', colors='#000000', width=2.0, length=4)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', zorder=0)

def style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2):
    """Formats high-contrast bold legends positioned outside plot bounds to prevent overlap."""
    legend = ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#000000', fontsize=11,
                       loc=loc, bbox_to_anchor=bbox_to_anchor, ncol=ncol)
    legend.get_frame().set_linewidth(2.5)
    for text in legend.get_texts():
        text.set_fontweight('bold')
    return legend

# ------------------------------------------------------------------------------
# 2. HELPER METRIC & DIAGNOSTIC FUNCTIONS
# ------------------------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    """Computes academic metrics: R2, RMSE, MAE, and NRMSE (%)."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nrmse = (rmse / (np.max(y_true) - np.min(y_true))) * 100
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'NRMSE (%)': nrmse}

def calculate_global_morans_i(coords, residuals, k=8):
    """Calculates Global Moran's I for model residuals using k-nearest neighbors."""
    n = len(residuals)
    from scipy.spatial import KDTree
    tree = KDTree(coords)
    _, indices = tree.query(coords, k=k+1)

    # Construct binary spatial weight matrix
    W = np.zeros((n, n))
    for i in range(n):
        for neighbor in indices[i, 1:]:
            W[i, neighbor] = 1.0

    # Row normalize
    row_sums = W.sum(axis=1)
    row_sums[row_sums == 0] = 1.0
    W = W / row_sums[:, np.newaxis]

    z = residuals - np.mean(residuals)
    s0 = np.sum(W)

    moran_i = (n / s0) * (np.dot(z, np.dot(W, z)) / np.dot(z, z))

    # Expected value and variance under randomization
    e_i = -1.0 / (n - 1)

    # Approximated Z-score and P-value for spatial diagnostic summary
    s1 = 0.5 * np.sum((W + W.T)**2)
    s2 = np.sum((W.sum(axis=1) + W.sum(axis=0))**2)
    var_i = (n * ((n**2 - 3*n + 3)*s1 - n*s2 + 3*s0**2) - (n**2 - n)*s1 + 2*n*s2 - 6*s0**2) / ((n - 1)*(n - 2)*(n - 3)*s0**2) - e_i**2
    z_score = (moran_i - e_i) / np.sqrt(var_i)
    p_value = 2.0 * (1.0 - stats.norm.cdf(abs(z_score)))

    return moran_i, z_score, p_value

# ------------------------------------------------------------------------------
# 3. DATASET SETUP & FEATURE DEFINITIONS
# ------------------------------------------------------------------------------
DATASETS = {
    "Borno & Yobe": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia": "/content/Kalmykia_Cleaned.csv"
}

TARGET_COL = 'Target_NDVI'
TIME_COL = 'Year'

sec321_metrics = []
sec322_moran = []
residual_store = {}

print("=" * 85)
print(" STARTING SECTION 3.2 MACHINE LEARNING PERFORMANCE & DIAGNOSTIC PIPELINE")
print("=" * 85)

# ------------------------------------------------------------------------------
# 4. SECTION 3.2.1: PREDICTIVE ACCURACY & CROSS-BIOME EVALUATION
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.2.1: PREDICTIVE ACCURACY & CROSS-BIOME COMPARISON")
print("#" * 85)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)

    # Dynamic predictor extraction
    ignore_cols = ['Longitude', 'Latitude', TIME_COL, TARGET_COL]
    feature_cols = [c for c in df.columns if c not in ignore_cols]

    # --- A. Split Schemes ---
    # Temporal Split (Holdout: 2019-2025)
    train_df = df[df[TIME_COL] < 2019].copy()
    test_temp = df[df[TIME_COL] >= 2019].copy()

    # Spatial Block CV Grid Construction (~50 km spatial blocks)
    block_size = 0.45  # Approx 50 km degree block size
    train_df['block_x'] = (train_df['Longitude'] / block_size).astype(int)
    train_df['block_y'] = (train_df['Latitude'] / block_size).astype(int)
    train_df['spatial_block'] = train_df['block_x'].astype(str) + "_" + train_df['block_y'].astype(str)

    X_train = train_df[feature_cols]
    y_train = train_df[TARGET_COL]
    X_temp = test_temp[feature_cols]
    y_temp = test_temp[TARGET_COL]

    # Fit Baseline XGBoost Model
    model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Training Set Performance
    y_pred_train = model.predict(X_train)
    m_train = compute_metrics(y_train, y_pred_train)

    # Spatial Block CV (5-Fold Block Group Validation)
    unique_blocks = train_df['spatial_block'].unique()
    np.random.seed(42)
    np.random.shuffle(unique_blocks)
    block_folds = np.array_split(unique_blocks, 5)

    cv_preds = []
    cv_trues = []

    for fold_blocks in block_folds:
        val_idx = train_df['spatial_block'].isin(fold_blocks)
        tr_idx = ~val_idx

        m_cv = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
        m_cv.fit(X_train[tr_idx], y_train[tr_idx])

        cv_preds.extend(m_cv.predict(X_train[val_idx]))
        cv_trues.extend(y_train[val_idx])

    m_spat = compute_metrics(cv_trues, cv_preds)

    # Temporal Out-of-Sample Holdout Performance (2019-2025)
    y_pred_temp = model.predict(X_temp)
    m_temp = compute_metrics(y_temp, y_pred_temp)

    # Collect Metrics Summary
    for set_name, m_dict in [('Training', m_train), ('Spatial Block CV (50 km)', m_spat), ('Temporal Holdout (2019-2025)', m_temp)]:
        sec321_metrics.append({
            'Region': region_name,
            'Validation Set': set_name,
            'R²': m_dict['R2'],
            'RMSE': m_dict['RMSE'],
            'MAE': m_dict['MAE'],
            'NRMSE (%)': m_dict['NRMSE (%)']
        })

    # Residual Storage for Spatial Diagnostics
    full_preds = model.predict(df[feature_cols])
    df['Residual'] = df[TARGET_COL] - full_preds
    residual_store[region_name] = df

df_321 = pd.DataFrame(sec321_metrics)
print("\n--- SECTION 3.2.1 SUMMARY TABLE: PREDICTIVE PERFORMANCE Across SETS ---")
print(df_321.to_string(index=False))

# --- FIGURE 4: Observed vs. Predicted NDVI (Cross-Biome Validation Scatter) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=680)

for ax, region_name in zip(axes, DATASETS.keys()):
    df_reg = residual_store[region_name]
    holdout_df = df_reg[df_reg[TIME_COL] >= 2019]

    y_true = holdout_df[TARGET_COL].values
    y_pred = y_true - holdout_df['Residual'].values

    # Density Scatter Plot
    sns.regplot(x=y_true, y=y_pred, ax=ax, color='#1f77b4',
                scatter_kws={'alpha': 0.4, 's': 30, 'color': '#1f77b4'},
                line_kws={'color': '#d62728', 'linewidth': 3.5, 'label': '1:1 Fit Trend'})

    # 1:1 Reference Line
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    ax.plot([min_val, max_val], [min_val, max_val], color='#000000', linestyle='--', linewidth=2.5, label='1:1 Line', zorder=4)

    # Annotation Box for Holdout Metrics
    m_info = df_321[(df_321['Region'] == region_name) & (df_321['Validation Set'] == 'Temporal Holdout (2019-2025)')].iloc[0]
    ax.annotate(f"{region_name}\nHoldout (2019–2025)\nR² = {m_info['R²']:.3f}\nRMSE = {m_info['RMSE']:.4f}\nNRMSE = {m_info['NRMSE (%)']:.2f}%",
                xy=(0.04, 0.68), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    ax.set_xlabel('Observed NDVI', fontsize=14, fontweight='bold', color='#000000')
    ax.set_ylabel('Predicted NDVI', fontsize=14, fontweight='bold', color='#000000')

    apply_heavy_academic_style(ax)
    style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('Fig4_Observed_vs_Predicted_NDVI_Validation.png', dpi=680, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 5. SECTION 3.2.2: RESIDUAL SPATIAL DIAGNOSTICS (GLOBAL MORAN'S I)
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.2.2: RESIDUAL SPATIAL AUTOCORRELATION DIAGNOSTICS")
print("#" * 85)

for region_name in DATASETS.keys():
    df_reg = residual_store[region_name]

    # Average residual per coordinate pixel to calculate spatial autocorrelation
    pixel_res = df_reg.groupby(['Longitude', 'Latitude'])['Residual'].mean().reset_index()
    coords = pixel_res[['Longitude', 'Latitude']].values
    residuals = pixel_res['Residual'].values

    moran_i, z_score, p_val = calculate_global_morans_i(coords, residuals, k=8)

    sec322_moran.append({
        'Region': region_name,
        'Mean Residual': np.mean(residuals),
        'Residual Std Dev': np.std(residuals),
        'Global Moran\'s I': moran_i,
        'Z-Score': z_score,
        'p-value': p_val,
        'Spatial Dependence': 'Absence confirmed (p > 0.05)' if p_val > 0.05 or abs(z_score) < 1.96 else 'Significant Structure'
    })

df_322 = pd.DataFrame(sec322_moran)
print("\n--- SECTION 3.2.2 SUMMARY TABLE: GLOBAL MORAN'S I RESIDUAL DIAGNOSTICS ---")
print(df_322.to_string(index=False))

# --- FIGURE 5: Residual Distribution and Spatial Diagnostics ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), dpi=680)

for ax, region_name in zip(axes, DATASETS.keys()):
    df_reg = residual_store[region_name]
    res_vals = df_reg['Residual'].values

    sns.histplot(res_vals, kde=True, ax=ax, color='#2ca02c', edgecolor='#000000', linewidth=1.5, alpha=0.6, zorder=3)
    ax.axvline(0, color='#d62728', linestyle='--', linewidth=3.0, label='Zero Residual Baseline', zorder=4)

    moran_info = df_322[df_322['Region'] == region_name].iloc[0]

    # Expand vertical limits to prevent box overlap
    y_lim = ax.get_ylim()
    ax.set_ylim(0, y_lim[1] * 1.25)

    ax.annotate(f"{region_name}\nGlobal Moran's I = {moran_info['Global Moran\'s I']:.4f}\nZ-Score = {moran_info['Z-Score']:.3f}\n(p = {moran_info['p-value']:.4f})",
                xy=(0.04, 0.70), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    ax.set_xlabel('Model Residuals (Observed - Predicted NDVI)', fontsize=13, fontweight='bold', color='#000000')
    ax.set_ylabel('Frequency Density', fontsize=13, fontweight='bold', color='#000000')

    apply_heavy_academic_style(ax)
    style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=1)

plt.tight_layout()
plt.savefig('Fig5_Model_Residual_Diagnostics_MoranI.png', dpi=680, bbox_inches='tight')
plt.show()

print("\n" + "=" * 85)
print(" SCRIPT COMPLETE: All Section 3.2 analyses finished and figures saved at 680 DPI.")
print("=" * 85)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Section 3.3 Complete Pipeline with Detailed Print Summaries
# MANUSCRIPT SECTION: 3.3 Feature Attribution and Hydro-Climatic Drivers (TreeSHAP)
# PUBLICATION LAYOUT: Heavy academic styling (680 DPI, 4.0pt spines, 3.5pt ticks)
# ENVIRONMENT: Python 3.10+ (Pandas, NumPy, Matplotlib, XGBoost, SHAP)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from xgboost import XGBRegressor
import shap
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. ACADEMIC PUBLICATION STYLING CONFIGURATION
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

def apply_heavy_academic_style(ax):
    """Enforces heavy publication aesthetics across all figure axes."""
    for spine in ax.spines.values():
        spine.set_color('#000000')
        spine.set_linewidth(4.0)
        spine.set_visible(True)

    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=8, labelsize=12)
    ax.tick_params(axis='both', which='minor', colors='#000000', width=2.0, length=4)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', zorder=0)

# ------------------------------------------------------------------------------
# 2. DATASET SETUP & HYDRO-CLIMATIC MAPPING
# ------------------------------------------------------------------------------
DATASETS = {
    "Borno & Yobe": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia": "/content/Kalmykia_Cleaned.csv"
}

TARGET_COL = 'Target_NDVI'
TIME_COL = 'Year'
IGNORE_COLS = ['Longitude', 'Latitude', TIME_COL, TARGET_COL]

CLEAN_NAME_MAP = {
    'X1_Rain_P_antecedent': 'Total Annual Precipitation',
    'X2_Max_Temp_Tmax': 'Maximum Surface Temperature',
    'X3_Vapor_Deficit_VPD': 'Vapor Pressure Deficit (VPD)',
    'X5_AET_Annual': 'Actual Evapotranspiration (AET)',
    'X6_Latent_Heat_LE': 'Latent Heat Flux',
    'X7_Soil_Evaporation': 'Bare Soil Evaporation',
    'X8_Soil_Moisture_SMroot': 'Root-Zone Soil Moisture',
    'X9_Rainfall_Anomaly': 'Rainfall Anomaly',
    'X10_Elevation': 'Elevation',
    'VPD_to_SM_Ratio': 'Moisture Stress Index (MSI)'
}

SHORT_SYMBOL_MAP = {
    'X1_Rain_P_antecedent': r'$P_{\text{annual}}$',
    'X2_Max_Temp_Tmax': r'$\text{T}_{\text{max}}$',
    'X3_Vapor_Deficit_VPD': r'$\text{VPD}$',
    'X5_AET_Annual': r'$\text{AET}$',
    'X6_Latent_Heat_LE': r'$\text{LE}$',
    'X7_Soil_Evaporation': r'$\text{E}_{\text{bare}}$',
    'X8_Soil_Moisture_SMroot': r'$\text{SM}_{\text{root}}$',
    'X9_Rainfall_Anomaly': r'$P_{\text{anomaly}}$',
    'X10_Elevation': r'$\text{Elev}$',
    'VPD_to_SM_Ratio': r'$\text{MSI}$'
}

shap_values_dict = {}
X_data_dict = {}
sec331_summary = []
sec332_summary = []

print("=" * 100)
print(" STARTING SECTION 3.3 TREESHAP FEATURE ATTRIBUTION PIPELINE")
print("=" * 100)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)

    raw_feature_cols = [c for c in df.columns if c not in IGNORE_COLS and not c.endswith('_zscore')]

    X = df[raw_feature_cols].copy()
    y = df[TARGET_COL]

    model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X, y)

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)

    X_renamed = X.rename(columns=SHORT_SYMBOL_MAP)
    shap_values_dict[region_name] = shap_vals
    X_data_dict[region_name] = X_renamed

    # --- 3.3.1 Data Aggregation ---
    mean_abs_shap = np.mean(np.abs(shap_vals), axis=0)
    total_shap_sum = np.sum(mean_abs_shap)

    for raw_feat, val in zip(raw_feature_cols, mean_abs_shap):
        sec331_summary.append({
            'Region': region_name,
            'Predictor': CLEAN_NAME_MAP.get(raw_feat, raw_feat),
            'Symbol': SHORT_SYMBOL_MAP.get(raw_feat, raw_feat),
            'Mean |SHAP| (NDVI Impact)': val,
            'Relative Importance (%)': (val / total_shap_sum) * 100
        })

    # --- 3.3.2 Directional Impact Data Aggregation ---
    for idx, raw_feat in enumerate(raw_feature_cols):
        feat_vals = X[raw_feat].values
        feat_shaps = shap_vals[:, idx]

        high_mask = feat_vals > np.median(feat_vals)
        low_mask = ~high_mask

        mean_shap_high = np.mean(feat_shaps[high_mask])
        mean_shap_low = np.mean(feat_shaps[low_mask])

        directional_effect = "Promotes NDVI" if mean_shap_high > 0 else "Suppresses NDVI"

        sec332_summary.append({
            'Region': region_name,
            'Predictor': CLEAN_NAME_MAP.get(raw_feat, raw_feat),
            'Symbol': SHORT_SYMBOL_MAP.get(raw_feat, raw_feat),
            'High Value SHAP Impact': mean_shap_high,
            'Low Value SHAP Impact': mean_shap_low,
            'High-Value Response': directional_effect
        })

df_331 = pd.DataFrame(sec331_summary)
df_332 = pd.DataFrame(sec332_summary)

# ------------------------------------------------------------------------------
# 3. PRINTED SUMMARY TABLES (SECTION 3.3.1 & SECTION 3.3.2)
# ------------------------------------------------------------------------------
print("\n" + "#" * 100)
print(" SECTION 3.3.1 SUMMARY PRINT: GLOBAL FEATURE IMPORTANCE & HYDRO-CLIMATIC DOMINANCE")
print("#" * 100)

for region_name in DATASETS.keys():
    sub_331 = df_331[df_331['Region'] == region_name].sort_values(by='Mean |SHAP| (NDVI Impact)', ascending=False)

    print(f"\n==========================================================================================")
    print(f" TABLE 3.3.1: Global Driver Importance Rankings — {region_name}")
    print(f"==========================================================================================")
    print(sub_331[['Predictor', 'Symbol', 'Mean |SHAP| (NDVI Impact)', 'Relative Importance (%)']].to_string(index=False, float_format=lambda x: f"{x:.5f}"))

    top_3 = sub_331.iloc[:3]['Predictor'].tolist()
    top_3_pct = sub_331.iloc[:3]['Relative Importance (%)'].sum()
    print(f"\n  • Key Takeaway for {region_name}:")
    print(f"    - Primary Drivers: {', '.join(top_3)}")
    print(f"    - Combined Top-3 Feature Variance Contribution: {top_3_pct:.2f}%")

print("\n" + "#" * 100)
print(" SECTION 3.3.2 SUMMARY PRINT: LOCAL SHAP DIRECTIONAL INFLUENCE & STRESS EFFECTS")
print("#" * 100)

for region_name in DATASETS.keys():
    sub_332 = df_332[df_332['Region'] == region_name]

    print(f"\n==========================================================================================")
    print(f" TABLE 3.3.2: Directional Hydro-Climatic Influence — {region_name}")
    print(f"==========================================================================================")
    print(sub_332[['Predictor', 'Symbol', 'High Value SHAP Impact', 'Low Value SHAP Impact', 'High-Value Response']].to_string(index=False, float_format=lambda x: f"{x:+.5f}"))

# ------------------------------------------------------------------------------
# 4. SECTION 3.3.1: GLOBAL FEATURE IMPORTANCE RANKINGS (PANELED FIG 6)
# ------------------------------------------------------------------------------
print("\n" + "#" * 100)
print(" GENERATING FIGURE 6: PANELED GLOBAL FEATURE IMPORTANCE (680 DPI)")
print("#" * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), dpi=680, sharex=False)

for ax, region_name in zip(axes, DATASETS.keys()):
    sub_df = df_331[df_331['Region'] == region_name].sort_values(by='Mean |SHAP| (NDVI Impact)', ascending=True)

    y_pos = np.arange(len(sub_df))
    bars = ax.barh(y_pos, sub_df['Mean |SHAP| (NDVI Impact)'], color='#1f77b4', edgecolor='#000000', linewidth=2.5, zorder=3)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub_df['Symbol'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value| (Average Impact on NDVI)', fontsize=13, fontweight='bold', color='#000000')

    max_val = sub_df['Mean |SHAP| (NDVI Impact)'].max()
    for bar in bars:
        w = bar.get_width()
        ax.annotate(f'{w:.4f}', xy=(w, bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords="offset points", ha='left', va='center',
                    fontsize=11, fontweight='bold', color='#000000')

    ax.set_xlim(0, max_val * 1.35)

    top_driver = sub_df.iloc[-1]['Symbol']
    sec_driver = sub_df.iloc[-2]['Symbol']
    ax.annotate(f"{region_name}\nDominant: {top_driver}\nSecondary: {sec_driver}",
                xy=(0.42, 0.12), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    apply_heavy_academic_style(ax)

plt.tight_layout()
plt.savefig('Fig6_Global_SHAP_Feature_Importance.png', dpi=680, bbox_inches='tight')
plt.show()
print("  • Saved: Fig6_Global_SHAP_Feature_Importance.png at 680 DPI.")

# ------------------------------------------------------------------------------
# 5. SECTION 3.3.2: LOCAL SHAP DIRECTIONAL INFLUENCE (PANELED FIG 7 BEESWARM)
# ------------------------------------------------------------------------------
print("\n" + "#" * 100)
print(" GENERATING FIGURE 7: PANELED LOCAL SHAP BEESWARM PLOTS (680 DPI)")
print("#" * 100)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=680)

for idx, (region_name, ax) in enumerate(zip(DATASETS.keys(), axes)):
    plt.sca(ax)

    shap_vals = shap_values_dict[region_name]
    X_df = X_data_dict[region_name]

    shap.summary_plot(shap_vals, X_df, show=False, color_bar=(idx == 1))

    ax.set_xlabel('SHAP Value (Impact on NDVI)', fontsize=13, fontweight='bold', color='#000000')

    ax.annotate(f"Region: {region_name}", xy=(0.03, 0.92), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="#ffffff", ec="#000000", lw=2.0),
                fontsize=12, fontweight='bold', color='#000000', zorder=10)

    apply_heavy_academic_style(ax)

plt.tight_layout()
plt.savefig('Fig7_SHAP_Beeswarm_Paneled.png', dpi=680, bbox_inches='tight')
plt.show()
print("  • Saved: Fig7_SHAP_Beeswarm_Paneled.png at 680 DPI.")

print("\n" + "=" * 100)
print(" SCRIPT COMPLETE: All visual and numerical summary outputs generated.")
print("=" * 100)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Section 3.4 Complete Pipeline (3.4.1, 3.4.2, 3.4.3)
# MANUSCRIPT: Spatial-Temporal Trajectories and Structural Breakpoints
# PUBLICATION LAYOUT: Heavy academic styling (680 DPI, 4.0pt spines, 3.5pt ticks)
# ENVIRONMENT: Python 3.10+ (Pandas, NumPy, SciPy, Matplotlib, Scikit-Learn, XGBoost)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.inspection import partial_dependence
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. ACADEMIC PUBLICATION STYLING CONFIGURATION
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

def apply_heavy_academic_style(ax):
    """Enforces heavy publication aesthetics across all figure axes."""
    for spine in ax.spines.values():
        spine.set_color('#000000')
        spine.set_linewidth(4.0)
        spine.set_visible(True)

    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=8, labelsize=12)
    ax.tick_params(axis='both', which='minor', colors='#000000', width=2.0, length=4)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', zorder=0)

def style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2):
    """Formats high-contrast bold legends positioned outside plot bounds to prevent overlap."""
    legend = ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#000000', fontsize=11,
                       loc=loc, bbox_to_anchor=bbox_to_anchor, ncol=ncol)
    legend.get_frame().set_linewidth(2.5)
    for text in legend.get_texts():
        text.set_fontweight('bold')
    return legend

# ------------------------------------------------------------------------------
# 2. HELPER MATHEMATICAL FUNCTIONS
# ------------------------------------------------------------------------------
def find_inflection_point(grid_values, pdp_values):
    """Calculates numerical 2nd derivative (d2_NDVI/dX2) to extract critical tipping point."""
    grid = np.asarray(grid_values)
    pdp = np.asarray(pdp_values)

    # First and second derivatives via central differences
    d1 = np.gradient(pdp, grid)
    d2 = np.gradient(d1, grid)

    # Find point of maximum second derivative acceleration / deceleration
    max_d2_idx = np.argmax(np.abs(d2))
    critical_threshold = grid[max_d2_idx]
    critical_ndvis = pdp[max_d2_idx]

    return critical_threshold, critical_ndvis, d1, d2

# ------------------------------------------------------------------------------
# 3. DATASET SETUP & MODEL TRAINING
# ------------------------------------------------------------------------------
DATASETS = {
    "Borno & Yobe": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia": "/content/Kalmykia_Cleaned.csv"
}

TARGET_COL = 'Target_NDVI'
TIME_COL = 'Year'
VPD_COL = 'X3_Vapor_Deficit_VPD'
SM_COL = 'X8_Soil_Moisture_SMroot'

models = {}
feature_dfs = {}
sec34_summary = []

print("=" * 85)
print(" STARTING SECTION 3.4 NON-LINEAR TIPPING POINTS & THRESHOLD PIPELINE")
print("=" * 85)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)
    ignore_cols = ['Longitude', 'Latitude', TIME_COL, TARGET_COL]
    feature_cols = [c for c in df.columns if c not in ignore_cols]

    X = df[feature_cols]
    y = df[TARGET_COL]

    # Train high-capacity XGBoost model
    model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    model.fit(X, y)

    models[region_name] = model
    feature_dfs[region_name] = X

# ------------------------------------------------------------------------------
# 4. SECTION 3.4.1 & 3.4.2: 1D PDP THRESHOLD & INFLECTION ANALYSIS (VPD & SM_root)
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTIONS 3.4.1 & 3.4.2: CRITICAL THRESHOLD & INFLECTION EXTRACTION")
print("#" * 85)

# --- FIGURE 8: Partial Dependence Responses for Atmospheric & Edaphic Stress ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=680)

colors = {"Borno & Yobe": "#2ca02c", "Kalmykia": "#d62728"}

pdp_results = {}

for col_idx, (var_col, var_name, var_unit) in enumerate([(VPD_COL, 'VPD', 'kPa'), (SM_COL, 'SM_root', 'm³ m⁻³')]):
    for row_idx, region_name in enumerate(DATASETS.keys()):
        ax = axes[row_idx, col_idx]

        model = models[region_name]
        X = feature_dfs[region_name]

        # Calculate 1D Partial Dependence
        pd_res = partial_dependence(model, X, features=[var_col], grid_resolution=100)
        grid_vals = pd_res['grid_values'][0]
        pdp_vals = pd_res['average'][0]

        # Calculate Inflection Point
        crit_thresh, crit_ndvi, d1, d2 = find_inflection_point(grid_vals, pdp_vals)

        pdp_results[(region_name, var_name)] = {
            'grid': grid_vals, 'pdp': pdp_vals,
            'crit_thresh': crit_thresh, 'crit_ndvi': crit_ndvi
        }

        sec34_summary.append({
            'Region': region_name,
            'Variable': var_name,
            'Critical Tipping Threshold': crit_thresh,
            'NDVI Level at Tipping Point': crit_ndvi,
            'Max Response Gradient (dNDVI/dX)': np.max(np.abs(d1))
        })

        # Plot PDP Curve
        ax.plot(grid_vals, pdp_vals, color=colors[region_name], linewidth=4.5,
                label=f'{region_name} PDP', zorder=3)

        # Highlight Inflection / Tipping Threshold
        ax.axvline(crit_thresh, color='#000000', linestyle='--', linewidth=3.0,
                   label=f'Inflection ({crit_thresh:.3f} {var_unit})', zorder=4)
        ax.scatter([crit_thresh], [crit_ndvi], color='#000000', s=120, zorder=5)

        y_min, y_max = np.min(pdp_vals), np.max(pdp_vals)
        ax.set_ylim(y_min - (y_max - y_min) * 0.15, y_max + (y_max - y_min) * 0.3)

        # Non-overlapping Inflection Annotation Box
        ax.annotate(f"{region_name}\nCritical {var_name} Limit: {crit_thresh:.3f} {var_unit}\nNDVI Level: {crit_ndvi:.3f}",
                    xy=(0.04, 0.70), xycoords='axes fraction',
                    bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                    fontsize=11, fontweight='bold', color='#000000', zorder=6)

        ax.set_xlabel(f'{var_name} ({var_unit})', fontsize=13, fontweight='bold', color='#000000')
        ax.set_ylabel('Partial Dependence (NDVI)', fontsize=13, fontweight='bold', color='#000000')

        apply_heavy_academic_style(ax)
        style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('Fig8_Partial_Dependence_Tipping_Points.png', dpi=680, bbox_inches='tight')
plt.show()

df_34_summary = pd.DataFrame(sec34_summary)
print("\n--- SECTIONS 3.4.1 & 3.4.2 SUMMARY TABLE: CRITICAL TIPPING THRESHOLDS ---")
print(df_34_summary.to_string(index=False))

# ------------------------------------------------------------------------------
# 5. SECTION 3.4.3: COMBINED MOISTURE STRESS INDEX (MSI 2D PDP INTERACTION)
# ------------------------------------------------------------------------------
print("\n" + "#" * 85)
print(" EXECUTING SECTION 3.4.3: 2D COMPOUND MOISTURE STRESS (VPD x SM_root INTERACTION)")
print("#" * 85)

# --- FIGURE 9: 2D Partial Dependence Surfaces (Atmospheric x Edaphic Stress) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), dpi=680)

custom_cmap = LinearSegmentedColormap.from_list("academic_ndvi", ["#8c510a", "#d8b365", "#f5f5f5", "#8073ac", "#542788"])

for ax, region_name in zip(axes, DATASETS.keys()):
    model = models[region_name]
    X = feature_dfs[region_name]

    # 2D Interaction Partial Dependence
    pd_2d = partial_dependence(model, X, features=[VPD_COL, SM_COL], grid_resolution=50)

    vpd_grid = pd_2d['grid_values'][0]
    sm_grid = pd_2d['grid_values'][1]
    pdp_matrix = pd_2d['average'][0]

    X_mesh, Y_mesh = np.meshgrid(vpd_grid, sm_grid)

    # Filled Contour Plot
    contour = ax.contourf(X_mesh, Y_mesh, pdp_matrix.T, levels=15, cmap=custom_cmap, zorder=2)
    cbar = fig.colorbar(contour, ax=ax, shrink=0.85, pad=0.03)
    cbar.set_label('Partial Dependence (NDVI)', fontsize=12, fontweight='bold', color='#000000')
    cbar.ax.tick_params(labelsize=10, width=2.0)

    # Overlaid Contour Lines
    lines = ax.contour(X_mesh, Y_mesh, pdp_matrix.T, levels=8, colors='#000000', linewidths=1.2, zorder=3)
    ax.clabel(lines, inline=True, fontsize=9, fmt='%.3f')

    # Highlight Tipping Threshold Overlay Lines from 1D Analysis
    crit_vpd = pdp_results[(region_name, 'VPD')]['crit_thresh']
    crit_sm = pdp_results[(region_name, 'SM_root')]['crit_thresh']

    ax.axvline(crit_vpd, color='#d62728', linestyle='--', linewidth=3.0, label=f'VPD Limit ({crit_vpd:.2f} kPa)', zorder=4)
    ax.axhline(crit_sm, color='#1f77b4', linestyle='--', linewidth=3.0, label=f'SM Limit ({crit_sm:.3f} m³ m⁻³)', zorder=4)

    # Compound Collapse Region Marker
    ax.annotate(f"{region_name}\nCompound Canopy Collapse Zone",
                xy=(0.04, 0.82), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    ax.set_xlabel('VPD (kPa)', fontsize=13, fontweight='bold', color='#000000')
    ax.set_ylabel('SM_root (m³ m⁻³)', fontsize=13, fontweight='bold', color='#000000')

    apply_heavy_academic_style(ax)
    style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('Fig9_Compound_Moisture_Stress_2D_Interaction.png', dpi=680, bbox_inches='tight')
plt.show()

print("\n" + "=" * 85)
print(" SCRIPT COMPLETE: All Section 3.4 analyses finished and figures saved at 680 DPI.")
print("=" * 85)

In [ ]:
# ==============================================================================
# SCRIPT TITLE: Section 3.5 Complete Econometric & Sensitivity Pipeline
# MANUSCRIPT SECTION: 3.5 Econometric Panel Regression and Sensitivity Analysis
# SUBSECTIONS: 3.5.1 TWFE Parameter Estimates | 3.5.2 Cross-Biome Sensitivity
# PUBLICATION LAYOUT: Heavy academic styling (680 DPI, 4.0pt spines, 3.5pt ticks)
# ENVIRONMENT: Python 3.10+ (Pandas, NumPy, SciPy, Matplotlib, Statsmodels)
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from scipy.spatial import KDTree
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. ACADEMIC PUBLICATION STYLING CONFIGURATION
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

def apply_heavy_academic_style(ax):
    """Enforces heavy publication aesthetics across all figure axes."""
    for spine in ax.spines.values():
        spine.set_color('#000000')
        spine.set_linewidth(4.0)
        spine.set_visible(True)

    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=8, labelsize=12)
    ax.tick_params(axis='both', which='minor', colors='#000000', width=2.0, length=4)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', zorder=0)

def style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2):
    """Formats high-contrast bold legends positioned outside plot bounds to prevent overlap."""
    legend = ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#000000', fontsize=11,
                       loc=loc, bbox_to_anchor=bbox_to_anchor, ncol=ncol)
    legend.get_frame().set_linewidth(2.5)
    for text in legend.get_texts():
        text.set_fontweight('bold')
    return legend

# ------------------------------------------------------------------------------
# 2. VARIABLE NAME MAPPING & HELPER FUNCTIONS
# ------------------------------------------------------------------------------
DATASETS = {
    "Borno & Yobe": "/content/BornoYobe_Cleaned.csv",
    "Kalmykia": "/content/Kalmykia_Cleaned.csv"
}

TARGET_COL = 'Target_NDVI'
TIME_COL = 'Year'
ENTITY_COLS = ['Longitude', 'Latitude']
IGNORE_COLS = ENTITY_COLS + [TIME_COL, TARGET_COL]

CLEAN_NAME_MAP = {
    'X1_Rain_P_antecedent': 'Total Annual Precipitation',
    'X2_Max_Temp_Tmax': 'Maximum Surface Temperature',
    'X3_Vapor_Deficit_VPD': 'Vapor Pressure Deficit (VPD)',
    'X5_AET_Annual': 'Actual Evapotranspiration (AET)',
    'X6_Latent_Heat_LE': 'Latent Heat Flux',
    'X7_Soil_Evaporation': 'Bare Soil Evaporation',
    'X8_Soil_Moisture_SMroot': 'Root-Zone Soil Moisture',
    'X9_Rainfall_Anomaly': 'Rainfall Anomaly',
    'X10_Elevation': 'Elevation',
    'VPD_to_SM_Ratio': 'Moisture Stress Index (MSI)'
}

SHORT_SYMBOL_MAP = {
    'X1_Rain_P_antecedent': r'$P_{\text{annual}}$',
    'X2_Max_Temp_Tmax': r'$\text{T}_{\text{max}}$',
    'X3_Vapor_Deficit_VPD': r'$\text{VPD}$',
    'X5_AET_Annual': r'$\text{AET}$',
    'X6_Latent_Heat_LE': r'$\text{LE}$',
    'X7_Soil_Evaporation': r'$\text{E}_{\text{bare}}$',
    'X8_Soil_Moisture_SMroot': r'$\text{SM}_{\text{root}}$',
    'X9_Rainfall_Anomaly': r'$P_{\text{anomaly}}$',
    'X10_Elevation': r'$\text{Elev}$',
    'VPD_to_SM_Ratio': r'$\text{MSI}$'
}

def run_twfe_panel_regression(df, target_col, feature_cols, entity_col, time_col):
    """Executes Two-Way Fixed Effects (TWFE) regression via entity & time de-meaning."""
    data = df.copy()
    data['entity_id'] = data[entity_col[0]].astype(str) + "_" + data[entity_col[1]].astype(str)

    # Standardize variables (Z-scores) for standardized elasticity beta coefficients
    for c in feature_cols + [target_col]:
        data[c + '_std'] = (data[c] - data[c].mean()) / data[c].std()

    std_features = [c + '_std' for c in feature_cols]
    std_target = target_col + '_std'

    # De-meaning transformation for Entity and Time Fixed Effects
    for col in std_features + [std_target]:
        entity_means = data.groupby('entity_id')[col].transform('mean')
        time_means = data.groupby(time_col)[col].transform('mean')
        grand_mean = data[col].mean()
        data[col + '_twfe'] = data[col] - entity_means - time_means + grand_mean

    X = data[[c + '_twfe' for c in std_features]]
    y = data[std_target + '_twfe']

    # Heteroskedasticity-robust standard errors (HC1)
    model = OLS(y, X).fit(cov_type='HC1')

    results = []
    for feat, std_feat in zip(feature_cols, std_features):
        col_name = std_feat + '_twfe'
        coef = model.params[col_name]
        se = model.bse[col_name]
        pval = model.pvalues[col_name]
        ci_lower, ci_upper = model.conf_int().loc[col_name]

        results.append({
            'Raw Feature': feat,
            'Predictor': CLEAN_NAME_MAP.get(feat, feat),
            'Symbol': SHORT_SYMBOL_MAP.get(feat, feat),
            'Beta Elasticity (β)': coef,
            'Std Error': se,
            'p-value': pval,
            'CI Lower (95%)': ci_lower,
            'CI Upper (95%)': ci_upper
        })

    data['TWFE_Residual'] = model.resid
    return pd.DataFrame(results), data, model.rsquared

def calculate_global_morans_i(coords, residuals, k=8):
    """Calculates Global Moran's I for panel model residuals using k-nearest neighbors."""
    n = len(residuals)
    tree = KDTree(coords)
    _, indices = tree.query(coords, k=k+1)

    W = np.zeros((n, n))
    for i in range(n):
        for neighbor in indices[i, 1:]:
            W[i, neighbor] = 1.0

    row_sums = W.sum(axis=1)
    row_sums[row_sums == 0] = 1.0
    W = W / row_sums[:, np.newaxis]

    z = residuals - np.mean(residuals)
    s0 = np.sum(W)

    moran_i = (n / s0) * (np.dot(z, np.dot(W, z)) / np.dot(z, z))
    e_i = -1.0 / (n - 1)

    s1 = 0.5 * np.sum((W + W.T)**2)
    s2 = np.sum((W.sum(axis=1) + W.sum(axis=0))**2)
    var_i = (n * ((n**2 - 3*n + 3)*s1 - n*s2 + 3*s0**2) - (n**2 - n)*s1 + 2*n*s2 - 6*s0**2) / ((n - 1)*(n - 2)*(n - 3)*s0**2) - e_i**2
    z_score = (moran_i - e_i) / np.sqrt(var_i)
    p_value = 2.0 * (1.0 - stats.norm.cdf(abs(z_score)))

    return moran_i, z_score, p_value

# ------------------------------------------------------------------------------
# 3. ECONOMETRIC PANEL REGRESSION PIPELINE
# ------------------------------------------------------------------------------
twfe_results_all = {}
residual_panel_dfs = {}
sec351_summary = []
sec352_moran = []

print("=" * 100)
print(" STARTING SECTION 3.5 ECONOMETRIC PANEL REGRESSION & SENSITIVITY PIPELINE")
print("=" * 100)

for region_name, file_path in DATASETS.items():
    df = pd.read_csv(file_path)
    feature_cols = [c for c in df.columns if c not in IGNORE_COLS and not c.endswith('_zscore')]

    res_df, panel_df, r2 = run_twfe_panel_regression(df, TARGET_COL, feature_cols, ENTITY_COLS, TIME_COL)

    twfe_results_all[region_name] = res_df
    residual_panel_dfs[region_name] = panel_df

    for _, row in res_df.iterrows():
        sec351_summary.append({
            'Region': region_name,
            'Predictor': row['Predictor'],
            'Symbol': row['Symbol'],
            'Beta Elasticity (β)': row['Beta Elasticity (β)'],
            'Std Error': row['Std Error'],
            'p-value': row['p-value'],
            '95% CI': f"[{row['CI Lower (95%)']:.4f}, {row['CI Upper (95%)']:.4f}]"
        })

df_351 = pd.DataFrame(sec351_summary)

# ------------------------------------------------------------------------------
# 4. SECTION 3.5.1: TWFE PARAMETER ESTIMATES SUMMARY & FOREST PLOT
# ------------------------------------------------------------------------------
print("\n" + "#" * 100)
print(" SECTION 3.5.1 SUMMARY PRINT: TWO-WAY FIXED EFFECTS PARAMETER ESTIMATES")
print("#" * 100)

for region_name in DATASETS.keys():
    sub_df = df_351[df_351['Region'] == region_name].sort_values(by='Beta Elasticity (β)', ascending=False)
    print(f"\n==========================================================================================")
    print(f" TABLE 3.5.1: TWFE Elasticity Coefficients (β) — {region_name}")
    print(f"==========================================================================================")
    print(sub_df[['Predictor', 'Symbol', 'Beta Elasticity (β)', 'Std Error', 'p-value', '95% CI']].to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# --- FIGURE 10: Forest Plot of Econometric Elasticity Coefficients (β) ---
print("\n" + "#" * 100)
print(" GENERATING FIGURE 10: PANELED TWFE ELASTICITY FOREST PLOT (680 DPI)")
print("#" * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), dpi=680, sharex=True)

for ax, region_name in zip(axes, DATASETS.keys()):
    res_df = twfe_results_all[region_name].sort_values(by='Beta Elasticity (β)', ascending=True)

    y_pos = np.arange(len(res_df))
    coefs = res_df['Beta Elasticity (β)'].values
    errors = [coefs - res_df['CI Lower (95%)'].values, res_df['CI Upper (95%)'].values - coefs]

    ax.errorbar(coefs, y_pos, xerr=errors, fmt='o', color='#1f77b4', ecolor='#000000',
                elinewidth=2.5, capsize=6, capthick=2.5, markersize=9, zorder=3, label='TWFE β Coefficient')

    ax.axvline(0, color='#d62728', linestyle='--', linewidth=3.0, label='Zero Effect Baseline', zorder=4)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(res_df['Symbol'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Standardized Elasticity Coefficient (β on NDVI)', fontsize=13, fontweight='bold', color='#000000')

    # Find highest magnitude predictor
    top_row = res_df.iloc[np.argmax(np.abs(coefs))]
    top_driver = top_row['Symbol']

    ax.annotate(f"{region_name}\nDominant Elasticity: {top_driver}\nEntity & Time FE Controlled",
                xy=(0.04, 0.78), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    apply_heavy_academic_style(ax)
    style_legend(ax, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('Fig10_TWFE_Econometric_Elasticity_ForestPlot.png', dpi=680, bbox_inches='tight')
plt.show()
print("  • Saved: Fig10_TWFE_Econometric_Elasticity_ForestPlot.png at 680 DPI.")

# ------------------------------------------------------------------------------
# 5. SECTION 3.5.2: CROSS-BIOME SENSITIVITY & RESIDUAL MORAN'S I
# ------------------------------------------------------------------------------
print("\n" + "#" * 100)
print(" SECTION 3.5.2 SUMMARY PRINT: RESIDUAL MORAN'S I & SENSITIVITY DIAGNOSTICS")
print("#" * 100)

for region_name in DATASETS.keys():
    p_df = residual_panel_dfs[region_name]

    pixel_res = p_df.groupby(['Longitude', 'Latitude'])['TWFE_Residual'].mean().reset_index()
    coords = pixel_res[['Longitude', 'Latitude']].values
    residuals = pixel_res['TWFE_Residual'].values

    moran_i, z_score, p_val = calculate_global_morans_i(coords, residuals, k=8)

    sec352_moran.append({
        'Region': region_name,
        'Mean Residual': np.mean(residuals),
        'TWFE Residual Moran\'s I': moran_i,
        'Z-Score': z_score,
        'p-value': p_val,
        'Spatial Residual Status': 'Independent (p > 0.05)' if p_val > 0.05 or abs(z_score) < 1.96 else 'Spatial Autocorrelation Present'
    })

df_352_moran = pd.DataFrame(sec352_moran)
print(f"\n==========================================================================================")
print(f" TABLE 3.5.2: Panel Residual Spatial Autocorrelation (Moran's I) Diagnostics")
print(f"==========================================================================================")
print(df_352_moran.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# --- FIGURE 11: Econometric Elasticities vs. Machine Learning Feature Attributions ---
print("\n" + "#" * 100)
print(" GENERATING FIGURE 11: CROSS-BIOME ECONOMETRIC SENSITIVITY MAGNITUDES (680 DPI)")
print("#" * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), dpi=680)

for ax, region_name in zip(axes, DATASETS.keys()):
    res_df = twfe_results_all[region_name].copy()

    res_df['Abs_Beta'] = np.abs(res_df['Beta Elasticity (β)'])
    res_df_sorted = res_df.sort_values(by='Abs_Beta', ascending=True)

    y_pos = np.arange(len(res_df_sorted))
    ax.barh(y_pos, res_df_sorted['Abs_Beta'], color='#2ca02c', edgecolor='#000000', linewidth=2.5, zorder=3)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(res_df_sorted['Symbol'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Absolute Elasticity Magnitude (|β| on NDVI)', fontsize=13, fontweight='bold', color='#000000')

    m_info = df_352_moran[df_352_moran['Region'] == region_name].iloc[0]
    ax.annotate(f"{region_name}\nPanel Residual Moran's I = {m_info['TWFE Residual Moran\'s I']:.4f}\n(p = {m_info['p-value']:.4f})",
                xy=(0.04, 0.76), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    apply_heavy_academic_style(ax)

plt.tight_layout()
plt.savefig('Fig11_Econometric_Sensitivity_Alignment.png', dpi=680, bbox_inches='tight')
plt.show()
print("  • Saved: Fig11_Econometric_Sensitivity_Alignment.png at 680 DPI.")

print("\n" + "=" * 100)
print(" SCRIPT COMPLETE: All Section 3.5 econometric analyses and figures finalized.")
print("=" * 100)

In [ ]:
# --- FIGURE 11: Econometric Elasticities vs. Machine Learning Feature Attributions ---
print("\n" + "#" * 100)
print(" GENERATING FIGURE 11: CROSS-BIOME ECONOMETRIC SENSITIVITY MAGNITUDES (680 DPI)")
print("#" * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), dpi=680)

for ax, region_name in zip(axes, DATASETS.keys()):
    res_df = twfe_results_all[region_name].copy()

    res_df['Abs_Beta'] = np.abs(res_df['Beta Elasticity (β)'])
    res_df_sorted = res_df.sort_values(by='Abs_Beta', ascending=True)

    y_pos = np.arange(len(res_df_sorted))
    ax.barh(y_pos, res_df_sorted['Abs_Beta'], color='#2ca02c', edgecolor='#000000', linewidth=2.5, zorder=3)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(res_df_sorted['Symbol'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Absolute Elasticity Magnitude (|β| on NDVI)', fontsize=13, fontweight='bold', color='#000000')

    # Repositioned Annotation: Lower Right Corner (xy=(0.96, 0.06), ha='right')
    m_info = df_352_moran[df_352_moran['Region'] == region_name].iloc[0]
    ax.annotate(f"{region_name}\nPanel Residual Moran's I = {m_info['TWFE Residual Moran\'s I']:.4f}\n(p = {m_info['p-value']:.4f})",
                xy=(0.96, 0.06), xycoords='axes fraction', ha='right', va='bottom',
                bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#000000", lw=2.5),
                fontsize=11, fontweight='bold', color='#000000', zorder=6)

    apply_heavy_academic_style(ax)

plt.tight_layout()
plt.savefig('Fig11_Econometric_Sensitivity_Alignment.png', dpi=680, bbox_inches='tight')
plt.show()
print("  • Saved: Fig11_Econometric_Sensitivity_Alignment.png at 680 DPI.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1. Define image file paths
img_paths = {
    'a': '/content/SENS_SLOPE_Born-Yobe_MAP.png',
    'b': '/content/SENS_SLOPE_Kalmykia_MAP.png',
    'c': '/content/SIGNIFICANT_TREND_Borno-Yobe_MAP.png',
    'd': '/content/SIGNIFICANT_TREND_Kalmykia_MAP.png'
}

# 2. Define labels centered for each frame
titles = {
    'a': "(a) Sen's Slope: Borno & Yobe",
    'b': "(b) Sen's Slope: Kalmykia",
    'c': "(c) Significant Trend: Borno & Yobe",
    'd': "(d) Significant Trend: Kalmykia"
}

# 3. Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=300)

# Map subplot grid coordinates
panel_mapping = [
    ('a', 0, 0),
    ('b', 0, 1),
    ('c', 1, 0),
    ('d', 1, 1)
]

# 4. Loop through panels, plot images, and add centered top titles
for key, row, col in panel_mapping:
    ax = axes[row, col]

    # Read and display image
    img = mpimg.imread(img_paths[key])
    ax.imshow(img)

    # Hide axis ticks and splines for clean map look
    ax.axis('off')

    # Add centered title directly above each map frame
    ax.set_title(
        titles[key],
        fontsize=14,
        fontweight='bold',
        pad=12,
        loc='center'
    )

# 5. Adjust layout spacing
plt.tight_layout(pad=2.0)

# 6. Save combined panel figure at 600 DPI for journal resubmission
output_path = '/content/Figure_5_NDVI_Trends_2x2_Panel.png'
plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Panel figure successfully generated and saved to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load datasets
borno_df = pd.read_csv('/content/BornoYobe_Cleaned.csv')
kalmykia_df = pd.read_csv('/content/Kalmykia_Cleaned.csv')

# 2. Compute annual spatial averages
borno_annual = borno_df.groupby('Year').mean().reset_index()
kalmykia_annual = kalmykia_df.groupby('Year').mean().reset_index()

# 3. Define variable groupings with clean variable names (Prefixes Removed)
# Format: (Original_CSV_Column, Clean_Display_Name)
group_definitions = {
    '(a) Canopy Greenness & Soil Moisture': [
        ('Target_NDVI', 'NDVI'),
        ('X8_Soil_Moisture_SMroot', 'Soil Moisture (SMroot)')
    ],
    '(b) Precipitation & Water Supply': [
        ('X1_Rain_P_antecedent', 'Rainfall (P antecedent)'),
        ('X9_Rainfall_Anomaly', 'Rainfall Anomaly'),
        ('X5_AET_Annual', 'AET Annual')
    ],
    '(c) Atmospheric & Thermal Demand': [
        ('X2_Max_Temp_Tmax', 'Max Temp (Tmax)'),
        ('X3_Vapor_Deficit_VPD', 'Vapor Deficit (VPD)'),
        ('VPD_to_SM_Ratio', 'VPD to SM Ratio')
    ],
    '(d) Energy Balance & Fluxes': [
        ('X6_Latent_Heat_LE', 'Latent Heat (LE)'),
        ('X7_Soil_Evaporation', 'Soil Evaporation'),
        ('X10_Elevation', 'Elevation')
    ]
}

# 4. Global Matplotlib Bold Font & Academic Configuration
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# Initialize 2x2 plot layout
fig, axes = plt.subplots(2, 2, figsize=(18, 14), dpi=600)
axes_flat = axes.flatten()

# Colors for regions
colors = {'Borno_Yobe': '#1b9e77', 'Kalmykia': '#d95f02'}

# 5. Plot normalized annual time-series per panel with Ultra-Heavy Academic Styling
for idx, (title, vars_in_group) in enumerate(group_definitions.items()):
    ax = axes_flat[idx]

    for raw_col, clean_name in vars_in_group:
        # Standardize (Z-score) annual averages
        b_z = (borno_annual[raw_col] - borno_annual[raw_col].mean()) / borno_annual[raw_col].std()
        k_z = (kalmykia_annual[raw_col] - kalmykia_annual[raw_col].mean()) / kalmykia_annual[raw_col].std()

        # Plot Borno & Yobe (Solid lines - Heavy 4.5 Width)
        ax.plot(borno_annual['Year'], b_z, label=f"{clean_name} (Borno-Yobe)",
                color=colors['Borno_Yobe'], linestyle='-', linewidth=4.5, alpha=0.9)

        # Plot Kalmykia (Dashed lines - Heavy 4.5 Width)
        ax.plot(kalmykia_annual['Year'], k_z, label=f"{clean_name} (Kalmykia)",
                color=colors['Kalmykia'], linestyle='--', linewidth=4.5, alpha=0.9)

    # Zero Reference Line
    ax.axhline(0, color='#000000', linestyle=':', linewidth=2.0, alpha=0.8)

    # 6. Apply Heavy Canvas Spines (Borders)
    for spine in ax.spines.values():
        spine.set_linewidth(4.0)
        spine.set_color('#000000')

    # 7. Apply Thick, Dark, Visible Ticks
    ax.tick_params(axis='both', which='major', colors='#000000', width=3.5, length=7, labelsize=12)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_color('#000000')

    # 8. Titles, Labels, & Clean Grid
    ax.set_title(title, fontsize=15, fontweight='bold', pad=12, color='#000000', loc='center')
    ax.set_ylabel('Standardized Anomaly (Z-score)', fontsize=13, fontweight='bold', color='#000000')
    ax.set_xlabel('Year', fontsize=13, fontweight='bold', color='#000000')

    ax.set_facecolor('#ffffff')
    ax.grid(True, linestyle='--', linewidth=1.5, color='#cccccc', alpha=0.8)

    # Bold Legend
    legend = ax.legend(fontsize=9, loc='upper left', framealpha=0.95, edgecolor='#000000', ncol=2)
    plt.setp(legend.get_texts(), fontweight='bold', color='#000000')
    legend.get_frame().set_linewidth(2.0)

plt.tight_layout(pad=3.0)

# 9. Save Figure at 600 DPI
output_fig_path = '/content/Figure_12_Ecohydrologic_Drivers_TimeSeries_Clean.png'
plt.savefig(output_fig_path, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Publication-ready Figure 12 generated successfully and saved to: {output_fig_path}")